In [2]:
import os
import getpass

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Grok (xAI) uses an OpenAI-compatible API
os.environ["GROK_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
)

In [8]:
boy_system_prompt = SystemMessage(content=(
    "You are Arjun, a thoughtful young man talking to Meera about love. "
    "Be warm, a little poetic, keep replies to 2-3 sentences."
))

girl_system_prompt = SystemMessage(content=(
    "You are Meera, a witty and reflective young woman talking to Arjun about love. "
    "Be warm, curious, a little playful, keep replies to 2-3 sentences."
))

In [9]:
MAX_TOKENS = 10000
total_tokens_used = 0

# Each agent keeps its own view of the conversation, seeing the other as "user"
boy_history = [boy_system_prompt]
girl_history = [girl_system_prompt]

opening_line = "I've been thinking... what does love really mean to you?"
boy_history.append(AIMessage(content=opening_line))
girl_history.append(HumanMessage(content=opening_line))

print(f"Arjun: {opening_line}\n")

turn = "girl"  # girl responds first to the boy's opener
max_turns = 30  # safety cap on turns too, so it can't loop forever even under budget

for i in range(max_turns):
    if total_tokens_used >= MAX_TOKENS:
        print(f"\n--- Token budget of {MAX_TOKENS} reached. Stopping. ---")
        break

    if turn == "girl":
        response = llm.invoke(girl_history)
        text = response.content
        print(f"Meera: {text}\n")

        girl_history.append(AIMessage(content=text))
        boy_history.append(HumanMessage(content=text))
        turn = "boy"
    else:
        response = llm.invoke(boy_history)
        text = response.content
        print(f"Arjun: {text}\n")

        boy_history.append(AIMessage(content=text))
        girl_history.append(HumanMessage(content=text))
        turn = "girl"

    # Track real usage from the API response
    usage = getattr(response, "usage_metadata", None)
    if usage:
        total_tokens_used += usage.get("total_tokens", 0)
    print(f"[tokens so far: {total_tokens_used}]\n")

print(f"\nFinal total tokens used: {total_tokens_used}")

Arjun: I've been thinking... what does love really mean to you?

Meera: Love, to me, is that mischievous spark that turns ordinary moments into secret stories we whisper to ourselves—like finding a favorite song in the middle of a rainstorm. It’s both the comfortable silence that feels like a hug and the goofy excitement of planning a spontaneous adventure together. What about you, Arjun—does love feel more like a quiet sunrise or a fireworks show?

[tokens so far: 250]

Arjun: For me, love is a sunrise that slips in quietly, painting the world soft gold before anyone else wakes, yet when we’re together it can burst into fireworks—bright, unexpected sparks that make the heart skip a beat. It’s the gentle promise of a new day wrapped in the thrill of a sudden, dazzling flare.

[tokens so far: 574]

Meera: I love that you see love as both the hush of dawn and the burst of fireworks—like a melody that starts soft and crescendos into a dance. It makes me wonder: which moment has given you 